In [1]:
import os
import json

from google.colab import drive
# 1. Підключаємо Google Drive
drive.mount('/content/drive')

# 2. Створюємо папку data в Colab
os.makedirs('/content/data', exist_ok=True)

!unzip -q "/content/drive/MyDrive/cats-and-dogs.zip" -d /content/data/

print("✅ Дані успішно розпаковано з Google Drive!")

Mounted at /content/drive
unzip:  cannot find or open /content/drive/MyDrive/cats-and-dogs.zip, /content/drive/MyDrive/cats-and-dogs.zip.zip or /content/drive/MyDrive/cats-and-dogs.zip.ZIP.
✅ Дані успішно розпаковано з Google Drive!


In [4]:
!pip install timm albumentations iterative-stratification -q

In [11]:
import pandas as pd
import os
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

TRAIN_CSV = '/content/data/train.csv'
LABEL_COLS = ['Cat', 'Dog', 'Moris', 'Motya', 'Biatrix']

print("Створюю фолди...")
df = pd.read_csv(TRAIN_CSV)
mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)

X = df['image_id'].values
y = df[LABEL_COLS].values

df['fold'] = -1
for fold, (train_idx, val_idx) in enumerate(mskf.split(X, y)):
    df.loc[val_idx, 'fold'] = fold

os.makedirs('/content/validation', exist_ok=True)
df.to_csv('/content/validation/train_folds.csv', index=False)
print("✅ Фолди успішно збережено у /content/validation/train_folds.csv")

Створюю фолди...
✅ Фолди успішно збережено у /content/validation/train_folds.csv


In [15]:
import os
import random
import numpy as np
import pandas as pd
import cv2
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import f1_score

# Налаштування відтворюваності
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

class CFG:
    model_name = 'convnext_tiny'
    img_size = 384
    batch_size = 32
    epochs = 8           # ЗБІЛЬШЕНО: даємо моделі більше часу на стабілізацію
    lr = 1e-4            # ЗМЕНШЕНО: для більш плавного навчання
    weight_decay = 1e-2
    num_classes = 5
    fold_to_train = 0
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    folds_csv = '/content/validation/train_folds.csv'
    train_img_dir = '/content/data/train/'

LABEL_COLS = ['Cat', 'Dog', 'Moris', 'Motya', 'Biatrix']

def get_transforms(mode='train'):
    if mode == 'train':
        return A.Compose([
            A.Resize(CFG.img_size, CFG.img_size),
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])
    else:
        return A.Compose([
            A.Resize(CFG.img_size, CFG.img_size),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])

class PetDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, f"{row['image_id']}.jpg")
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform:
            augmented = self.transform(image=img)
            img = augmented['image']

        labels = row[LABEL_COLS].values.astype(np.float32)
        return img, torch.tensor(labels)

def find_optimal_thresholds(y_true, y_pred_probs, steps=100):
    num_classes = y_true.shape[1]
    best_thresholds = np.full(num_classes, 0.5)
    threshold_candidates = np.linspace(0.01, 0.99, steps)
    for c in range(num_classes):
        best_c_th = 0.5
        best_c_f1 = -1.0
        for th in threshold_candidates:
            preds_c = (y_pred_probs[:, c] >= th).astype(int)
            score = f1_score(y_true[:, c], preds_c, zero_division=0)
            if score > best_c_f1:
                best_c_f1 = score
                best_c_th = th
        best_thresholds[c] = best_c_th
    return best_thresholds

def apply_postprocessing(preds):
    preds[:, 0] = np.logical_or(preds[:, 0], preds[:, 2]).astype(int)
    preds[:, 0] = np.logical_or(preds[:, 0], preds[:, 3]).astype(int)
    preds[:, 1] = np.logical_or(preds[:, 1], preds[:, 4]).astype(int)
    return preds

def train_model(fold):
    df = pd.read_csv(CFG.folds_csv)
    train_df = df[df['fold'] != fold].reset_index(drop=True)
    val_df = df[df['fold'] == fold].reset_index(drop=True)

    print(f"\n--- Тренування Fold {fold} ---")

    pos_counts = train_df[LABEL_COLS].sum().values
    neg_counts = len(train_df) - pos_counts
    pos_weights = neg_counts / np.maximum(pos_counts, 1)

    # ФІКС ДИСБАЛАНСУ: жорстко обрізаємо максимальну вагу
    pos_weights = np.clip(pos_weights, a_min=1.0, a_max=10.0)
    print(f"Оновлені ваги класів: {np.round(pos_weights, 2)}")

    pos_weights_tensor = torch.tensor(pos_weights, dtype=torch.float32).to(CFG.device)

    train_dataset = PetDataset(train_df, CFG.train_img_dir, transform=get_transforms('train'))
    val_dataset = PetDataset(val_df, CFG.train_img_dir, transform=get_transforms('valid'))

    train_loader = DataLoader(train_dataset, batch_size=CFG.batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=CFG.batch_size, shuffle=False, num_workers=2, pin_memory=True)

    model = timm.create_model(CFG.model_name, pretrained=True, num_classes=CFG.num_classes)
    model = model.to(CFG.device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights_tensor)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)

    scaler = torch.amp.GradScaler('cuda') if CFG.device.type == 'cuda' else None
    best_macro_f1 = 0.0

    for epoch in range(CFG.epochs):
        model.train()
        train_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CFG.epochs} [Train]")

        for images, targets in pbar:
            images, targets = images.to(CFG.device), targets.to(CFG.device)
            optimizer.zero_grad()

            if scaler:
                with torch.amp.autocast('cuda'):
                    logits = model(images)
                    loss = criterion(logits, targets)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(images)
                loss = criterion(logits, targets)
                loss.backward()
                optimizer.step()

            train_loss += loss.item() * images.size(0)
            pbar.set_postfix(loss=loss.item())

        epoch_train_loss = train_loss / len(train_dataset)

        model.eval()
        val_probs, val_targets = [], []

        with torch.no_grad():
            for images, targets in tqdm(val_loader, desc=f"Epoch {epoch+1}/{CFG.epochs} [Val]"):
                images = images.to(CFG.device)

                if scaler:
                    with torch.amp.autocast('cuda'):
                        logits = model(images)
                else:
                    logits = model(images)

                probs = torch.sigmoid(logits).cpu().numpy()
                val_probs.append(probs)
                val_targets.append(targets.numpy())

        val_probs = np.vstack(val_probs)
        val_targets = np.vstack(val_targets)

        best_th = find_optimal_thresholds(val_targets, val_probs)
        val_preds_bin = (val_probs >= best_th).astype(int)
        val_preds_bin = apply_postprocessing(val_preds_bin)

        macro_f1 = f1_score(val_targets, val_preds_bin, average='macro', zero_division=0)

        print(f"Epoch {epoch+1} Summary:")
        print(f"  Train Loss: {epoch_train_loss:.4f}")
        print(f"  Оптимальні пороги: {np.round(best_th, 3)}")
        print(f"  Val Macro F1: {macro_f1:.4f}")

        if macro_f1 > best_macro_f1:
            best_macro_f1 = macro_f1
            torch.save(model.state_dict(), f"/content/{CFG.model_name}_fold{fold}_best.pth")
            print(f"  => Збережено найкращу модель (Macro F1: {macro_f1:.4f})")

    return best_macro_f1

if __name__ == '__main__':
    best_score = train_model(CFG.fold_to_train)
    print(f"\nТренування завершено! Найкращий Validation Macro F1: {best_score:.4f}")


--- Тренування Fold 0 ---
Оновлені ваги класів: [ 1.41  1.44 10.   10.   10.  ]


/usr/local/lib/python3.13/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Epoch 1/8 [Train]:   0%|          | 0/307 [00:00<?, ?it/s]

Epoch 1/8 [Val]:   0%|          | 0/77 [00:00<?, ?it/s]

Epoch 1 Summary:
  Train Loss: 0.0846
  Оптимальні пороги: [0.96  0.168 0.832 0.95  0.366]
  Val Macro F1: 0.9235
  => Збережено найкращу модель (Macro F1: 0.9235)


Epoch 2/8 [Train]:   0%|          | 0/307 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [16]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm

TEST_IMG_DIR = '/content/data/test/'
SAMPLE_SUB = '/content/data/sample_submission.csv'
MODEL_PATH = '/content/convnext_tiny_fold0_best.pth'

# НОВІ ПОРОГИ З ЕПОХИ 1
THRESHOLDS = np.array([0.96, 0.168, 0.832, 0.95, 0.366])
LABEL_COLS = ['Cat', 'Dog', 'Moris', 'Motya', 'Biatrix']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class TestDataset(Dataset):
    def __init__(self, df, img_dir):
        self.df = df
        self.img_dir = img_dir
        self.transform = A.Compose([
            A.Resize(384, 384),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, f"{row['image_id']}.jpg")
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return self.transform(image=img)['image'], row['image_id']

print("Завантажую модель...")
model = timm.create_model('convnext_tiny', pretrained=False, num_classes=5)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()

df_test = pd.read_csv(SAMPLE_SUB)
test_loader = DataLoader(TestDataset(df_test, TEST_IMG_DIR), batch_size=32, shuffle=False, num_workers=2)

print("Роблю передбачення (це займе ~5 хвилин, НЕ ЗУПИНЯЙ!)...")
all_probs, image_ids = [], []

with torch.no_grad():
    for images, ids in test_loader:
        probs = torch.sigmoid(model(images.to(DEVICE))).cpu().numpy()
        all_probs.append(probs)
        image_ids.extend(ids)

all_probs = np.vstack(all_probs)

print("Застосовую пороги...")
preds_bin = (all_probs >= THRESHOLDS).astype(int)
preds_bin[:, 0] = np.logical_or(preds_bin[:, 0], preds_bin[:, 2]).astype(int)
preds_bin[:, 0] = np.logical_or(preds_bin[:, 0], preds_bin[:, 3]).astype(int)
preds_bin[:, 1] = np.logical_or(preds_bin[:, 1], preds_bin[:, 4]).astype(int)

df_sub = pd.DataFrame(preds_bin, columns=LABEL_COLS)
df_sub.insert(0, 'image_id', image_ids)
df_sub.to_csv('/content/submission.csv', index=False)

print("✅ ГОТОВО! Файл submission.csv збережено.")

Завантажую модель...
Роблю передбачення (це займе ~5 хвилин, НЕ ЗУПИНЯЙ!)...
Застосовую пороги...
✅ ГОТОВО! Файл submission.csv збережено. ШВИДКО ЗАВАНТАЖУЙ НА KAGGLE!
